<a href="https://colab.research.google.com/github/Monikagupta5/ConfidenceCoach-AI/blob/master/Confidence%20Coach%20AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install fastapi uvicorn nest-asyncio pyngrok spacy transformers
!python -m spacy download en_core_web_sm
!pip install SpeechRecognition
!pip install pydub
!pip install python-dotenv
!pip install python-multipart



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.2/73.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 59.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.8/32.8 MB 51.5 MB/s eta 0:00:00


In [18]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, UploadFile
from pydantic import BaseModel
import spacy
import logging
import speech_recognition as sr
from typing import List
import re

# Load spaCy model for NLP tasks
nlp = spacy.load("en_core_web_sm")

# Define patterns for low-confidence language
LOW_CONFIDENCE_PATTERNS = [
    ("I just think", "Consider removing 'just' to make it more assertive."),
    ("I'm sorry", "Avoid excessive apologizing unless necessary."),
    ("maybe", "Try using more definitive words."),
    ("I could be wrong, but", "Consider removing the disclaimer to sound more confident."),
    ("I think", "Consider removing 'I think' to sound more confident."),
    ("perhaps", "Replace 'perhaps' with a more definitive word."),
    ("I can't do it", "Consider using positive language."),
    ("I'm not sure", "Avoid showing uncertainty. Use a confident statement instead."),
    ("unsure", "Replace with a more assertive phrase."),
    ("totally unsure", "Replace with a confident alternative."),
    ("we should consider", "Use 'we must' or 'we will' for assertiveness.")
]

# Enhanced passive voice detection function
def detect_passive_voice(doc):
    """
    Detects passive voice constructions in a spaCy Doc object.

    Args:
        doc: A spaCy Doc object.

    Returns:
        A list of suggestions indicating passive voice occurrences.
    """
    passive_suggestions = []
    for token in doc:
        # Check for passive construction
        if token.dep_ == "auxpass" and token.head.pos_ == "VERB":
            agent = [child for child in token.head.children if child.dep_ == "agent"]
            if agent:
                passive_suggestions.append(f"Passive voice detected: '{token.head.text}' with agent '{agent[0].text}'")
            else:
                passive_suggestions.append(f"Passive voice detected: '{token.head.text}'")
    return passive_suggestions

def analyze_text_spacy(text):
    """
    Analyzes text using spaCy and provides suggestions for improvement.

    Args:
        text: The input text.

    Returns:
        A dictionary containing confidence scores, suggestions, and highlighted text.
    """
    doc = nlp(text)
    passive_voice_suggestions = detect_passive_voice(doc)  # Call the updated function
    section_scores = []
    suggestions = []

    confidence_score = 5  # Full confidence to start with

    for sent in doc.sents:
        score = 5
        local_suggestions = []
        for pattern, suggestion in LOW_CONFIDENCE_PATTERNS:
            if re.search(re.escape(pattern), sent.text, re.IGNORECASE):
                local_suggestions.append((pattern, suggestion))
                score -= 1  # Deduct points for each pattern match

        section_scores.append({
            "text": sent.text,
            "confidence_score": score,
            "suggestions": local_suggestions
        })
        suggestions.extend(local_suggestions)

    # Overall confidence score is a balance of both ML and spaCy analysis
    overall_score = max(1, confidence_score - len(suggestions))
    return {
        "overall_confidence_score": overall_score,
        "section_scores": section_scores,
        "passive_voice_suggestions": passive_voice_suggestions,
        "highlighted_text": [str(ent) for ent in doc.ents],
        "suggestions": suggestions
    }

# Model and data setup for machine learning analysis
import random

def generate_synthetic_data():
    base_phrases = [
        ("I just think", "Underconfident"),
        ("I'm sorry", "Underconfident"),
        ("maybe", "Underconfident"),
        ("I could be wrong, but", "Underconfident"),
        ("I think", "Underconfident"),
        ("perhaps", "Underconfident"),
        ("I'm not sure", "Underconfident"),
        ("unsure", "Underconfident"),
        ("totally unsure", "Underconfident"),
        ("I can't do it", "Underconfident"),
        ("This is a good idea.", "Neutral"),
        ("I am the best", "Neutral"),
        ("I can do it", "Neutral"),
        ("Let's move forward with this strategy.", "Neutral"),
        ("It is possible", "Neutral"),
        ("Responses may delay", "Neutral"),
        ("Data steps for ML analysis", "Neutral")

    ]

    synthetic_data = []
    for phrase, label in base_phrases:
        for _ in range(50):
            random_suffix = f" {random.choice(['!', 'maybe?', 'surely.', 'variation'])}"
            variation = phrase + random_suffix
            synthetic_data.append((variation, label))

    return synthetic_data


data = generate_synthetic_data()
df = pd.DataFrame(data, columns=["Text", "Label"])
label_map = {"Underconfident": 0, "Neutral": 1}
df["Label"] = df["Label"].map(label_map)

# Feature extraction using TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df["Text"])
y = df["Label"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train logistic regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Function to classify text using ML model
def classify_text_ml(input_text):
    input_vec = vectorizer.transform([input_text])
    probabilities = model.predict_proba(input_vec)
    reverse_label_map = {v: k for k, v in label_map.items()}
    return {
        "label": reverse_label_map[np.argmax(probabilities)],
        "confidence_scores": {reverse_label_map[i]: prob for i, prob in enumerate(probabilities[0])}
    }

# Enhanced function to analyze text using spaCy and provide suggestions
def analyze_text_spacy(text):
    doc = nlp(text)
    passive_voice_suggestions = detect_passive_voice(doc)
    section_scores = []
    suggestions = []

    confidence_score = 5  # Full confidence to start with

    for sent in doc.sents:
        score = 5
        local_suggestions = []
        for pattern, suggestion in LOW_CONFIDENCE_PATTERNS:
            if pattern in sent.text:
                local_suggestions.append((pattern, suggestion))
                score -= 1  # Deduct points for each pattern match

        section_scores.append({
            "text": sent.text,
            "confidence_score": score,
            "suggestions": local_suggestions
        })
        suggestions.extend(local_suggestions)

    # Overall confidence score is a balance of both ML and spaCy analysis
    overall_score = max(1, confidence_score - len(suggestions))
    return {
        "overall_confidence_score": overall_score,
        "section_scores": section_scores,
        "passive_voice_suggestions": passive_voice_suggestions,
        "highlighted_text": [str(ent) for ent in doc.ents],
        "suggestions": suggestions
    }

# Transcribe audio to text function
def transcribe_speech(audio_file_path):
    recognizer = sr.Recognizer()
    try:
        with sr.AudioFile(audio_file_path) as source:
            audio = recognizer.record(source)
        return recognizer.recognize_google(audio)
    except sr.UnknownValueError:
        return "Unable to recognize speech. Please try again."
    except sr.RequestError as e:
        return f"Speech Recognition API unavailable: {e}"
    except Exception as e:
        logging.error(f"Error transcribing audio: {str(e)}")
        return f"Error: {str(e)}"


# FastAPI app definition
app = FastAPI()

# Define user input model
class UserInput(BaseModel):
    text: str

# Mock database for storing user revisions
user_revisions = {}

@app.post("/classify/")
def classify_text_api(input: UserInput):
    ml_result = classify_text_ml(input.text)
    spacy_result = analyze_text_spacy(input.text)
    return {
        "text": input.text,
        "classification": ml_result,
        "spacy_analysis": spacy_result
    }

@app.post("/transcribe/")
def transcribe_audio(file: UploadFile):
    text = transcribe_speech(file.file)
    return {"text": text}

@app.post("/submit/")
def submit_text(user_id: str, text: str):
    if user_id not in user_revisions:
        user_revisions[user_id] = []
    user_revisions[user_id].append(text)
    return {"message": "Text submitted successfully", "history": user_revisions[user_id]}

# Sample Test: Detecting passive voice
doc = nlp("The ball was thrown by the boy.")
print(detect_passive_voice(doc))


def analyze_text_spacy(text):
    doc = nlp(text)
    passive_voice_suggestions = detect_passive_voice(doc)
    section_scores = []
    suggestions = []

    confidence_score = 5  # Full confidence to start with

    for sent in doc.sents:
        score = 5
        local_suggestions = []
        for pattern, suggestion in LOW_CONFIDENCE_PATTERNS:
            if re.search(re.escape(pattern), sent.text, re.IGNORECASE):
                local_suggestions.append((pattern, suggestion))
                score -= 1  # Deduct points for each pattern match

        section_scores.append({
            "text": sent.text,
            "confidence_score": score,
            "suggestions": local_suggestions
        })
        suggestions.extend(local_suggestions)

    # Overall confidence score is a balance of both ML and spaCy analysis
    overall_score = max(1, confidence_score - len(suggestions))
    return {
        "overall_confidence_score": overall_score,
        "section_scores": section_scores,
        "passive_voice_suggestions": passive_voice_suggestions,
        "highlighted_text": [str(ent) for ent in doc.ents],
        "suggestions": suggestions
    }

    from difflib import SequenceMatcher

@app.post("/submit/")
def submit_text(user_id: str, text: str):
    if user_id not in user_revisions:
        user_revisions[user_id] = []

    # Compare with the last revision
    previous_text = user_revisions[user_id][-1] if user_revisions[user_id] else ""
    similarity = SequenceMatcher(None, previous_text, text).ratio()

    user_revisions[user_id].append(text)
    return {
        "message": "Text submitted successfully",
        "history": user_revisions[user_id],
        "similarity_with_previous": f"{similarity:.2f}"
    }


y_pred = model.predict(X_test)
print("Classification Report:")
print(classification_report(y_test, y_pred))







["Passive voice detected: 'thrown' with agent 'by'"]
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95        95
           1       0.96      0.92      0.94        75

    accuracy                           0.95       170
   macro avg       0.95      0.94      0.95       170
weighted avg       0.95      0.95      0.95       170



In [16]:
!ngrok config add-authtoken 2q58dFTb2DoAUxBGqQml8NBY3PV_7t2PVEx1y3uXGqriNDSYy

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
# Allow nested asyncio loops
nest_asyncio.apply()

# Set up ngrok
public_url = ngrok.connect(8000)
print(f"Public URL: {public_url}")

# Start the server
uvicorn.run(app, host="0.0.0.0", port=8000)


INFO:     Started server process [454]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://007a-34-27-175-141.ngrok-free.app" -> "http://localhost:8000"
INFO:     2003:fe:4702:e18d:b5ec:500:96a4:6e3e:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2003:fe:4702:e18d:b5ec:500:96a4:6e3e:0 - "GET /openapi.json HTTP/1.1" 200 OK


/usr/local/lib/python3.10/dist-packages/fastapi/openapi/utils.py:225: UserWarning: Duplicate Operation ID submit_text_submit__post for function submit_text
  warnings.warn(message, stacklevel=1)
ERROR:root:Error transcribing audio: Audio file could not be read as PCM WAV, AIFF/AIFF-C, or Native FLAC; check if file is corrupted or in another format


INFO:     2003:fe:4702:e18d:b5ec:500:96a4:6e3e:0 - "POST /transcribe/ HTTP/1.1" 200 OK


ERROR:root:Error transcribing audio: Audio file could not be read as PCM WAV, AIFF/AIFF-C, or Native FLAC; check if file is corrupted or in another format


INFO:     2003:fe:4702:e18d:b5ec:500:96a4:6e3e:0 - "POST /transcribe/ HTTP/1.1" 200 OK
INFO:     2003:fe:4702:e18d:b5ec:500:96a4:6e3e:0 - "POST /classify/ HTTP/1.1" 200 OK
